In [14]:
import os
import joblib
import numpy as np

from tqdm import tqdm

from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.model_selection import (
    StratifiedKFold,
    ParameterGrid
)

In [15]:
FEATURE_DIR = r"C:\Users\AmrAhmed\Desktop\Level 3\Second Term\Pattern Recognition\Project\Drowsiness-Project\feature_extraction\features_output"

X_train_path = os.path.join(FEATURE_DIR, "X_train_features.npy")
X_test_path  = os.path.join(FEATURE_DIR, "X_test_features.npy")
y_train_path = os.path.join(FEATURE_DIR, "y_train.npy")
y_test_path  = os.path.join(FEATURE_DIR, "y_test.npy")
scaler_path  = os.path.join(FEATURE_DIR, "scaler.pkl")
model_path   = os.path.join(FEATURE_DIR, "svm_model.pkl")

In [16]:
X_train = np.load(X_train_path)
X_test  = np.load(X_test_path)
y_train = np.load(y_train_path)
y_test  = np.load(y_test_path)

y_train = y_train.ravel()
y_test = y_test.ravel()

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (22205, 1282)
X_test shape : (5568, 1282)
y_train shape: (22205,)
y_test shape : (5568,)


In [17]:
scaler = joblib.load(scaler_path)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling done successfully.")

Scaling done successfully.


In [18]:
svm = SVC(class_weight="balanced")
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [19]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

grid_list = list(ParameterGrid(param_grid))

print("Total Configurations:", len(grid_list))

Total Configurations: 12


In [20]:
kf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [21]:
best_score = 0
best_params = None
best_model = None

results = []

for params in tqdm(grid_list, desc="Grid Search"):

    fold_scores = []

    base_model = SVC(
        **params,
        class_weight="balanced"
    )

    for train_idx, val_idx in tqdm(
        kf.split(X_train, y_train),
        total=kf.get_n_splits(),
        leave=False,
        desc=f"Folds {params}"
    ):
        X_tr = X_train[train_idx]
        X_val = X_train[val_idx]
        y_tr = y_train[train_idx]
        y_val = y_train[val_idx]

        model = clone(base_model)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)
        acc = accuracy_score(y_val, y_pred)

        fold_scores.append(acc)
    mean_score = np.mean(fold_scores)

    results.append({
        "params": params,
        "score": mean_score
    })
    print(f"\nParams: {params}")
    print(f"Mean Accuracy: {mean_score:.4f}")

    if mean_score > best_score:
        best_score = mean_score
        best_params = params
        best_model = clone(base_model)

Grid Search:   0%|          | 0/12 [00:00<?, ?it/s]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}:  33%|███▎      | 1/3 [00:51<01:42, 51.29s/it]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}:  67%|██████▋   | 2/3 [01:50<00:56, 56.13s/it]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}: 100%|██████████| 3/3 [02:44<00:00, 55.21s/it]
Grid Search:   8%|▊         | 1/12 [02:44<30:14, 164.94s/it]                                         


Params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Mean Accuracy: 0.9558



Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}:  33%|███▎      | 1/3 [02:39<05:19, 159.88s/it]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}:  67%|██████▋   | 2/3 [05:28<02:45, 165.00s/it]
Folds {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}: 100%|██████████| 3/3 [08:26<00:00, 170.88s/it]
Grid Search:  17%|█▋        | 2/12 [11:11<1:00:57, 365.77s/it]                                     


Params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Mean Accuracy: 0.8730



Folds {'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}:  33%|███▎      | 1/3 [00:53<01:47, 53.56s/it]
Folds {'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}:  67%|██████▋   | 2/3 [01:46<00:52, 52.93s/it]
Folds {'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}: 100%|██████████| 3/3 [02:36<00:00, 51.64s/it]
Grid Search:  25%|██▌       | 3/12 [13:47<40:30, 270.05s/it]                                        

...


Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Mean Accuracy: 0.9766



Folds {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}:  33%|███▎      | 1/3 [00:47<01:34, 47.44s/it]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}:  67%|██████▋   | 2/3 [01:37<00:48, 48.90s/it]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}: 100%|██████████| 3/3 [02:22<00:00, 47.38s/it]
Grid Search:  92%|█████████▏| 11/12 [56:13<04:02, 242.27s/it]                                      


Params: {'C': 10, 'gamma': 'auto', 'kernel': 'linear'}
Mean Accuracy: 0.9558



Folds {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}:   0%|          | 0/3 [00:00<?, ?it/s]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}:  33%|███▎      | 1/3 [06:25<12:50, 385.34s/it]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}:  67%|██████▋   | 2/3 [13:58<07:05, 425.31s/it]
Folds {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}: 100%|██████████| 3/3 [20:19<00:00, 405.04s/it]
Grid Search: 100%|██████████| 12/12 [1:16:32<00:00, 382.72s/it]                                  


Params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
Mean Accuracy: 0.5319


In [22]:
print("\nBest Parameters:", best_params)
print("Best CV Accuracy:", best_score)

best_model.fit(X_train, y_train)

print("\nFinal model trained.")


Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV Accuracy: 0.9765817068348829

Final model trained.


In [23]:
y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print("\nTest Accuracy:", test_accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Test Accuracy: 0.9832974137931034

Confusion Matrix:
[[2801   30]
 [  63 2674]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      2831
           1       0.99      0.98      0.98      2737

    accuracy                           0.98      5568
   macro avg       0.98      0.98      0.98      5568
weighted avg       0.98      0.98      0.98      5568



In [24]:
joblib.dump(best_model, "svm_model.pkl")

print(f"SVM model saved successfully at:\n{model_path}")

SVM model saved successfully at:
C:\Users\AmrAhmed\Desktop\Level 3\Second Term\Pattern Recognition\Project\Drowsiness-Project\feature_extraction\features_output\svm_model.pkl
